# Patient-level model comparison: Decision Tree, Random Forest and LightGBM

Notebook này so sánh ba model với XGBoost Sepsyd reproduce, dùng cùng preprocessing và patient-level folds:

- Tuning 5-fold một lần: tìm một `best_config` cho mỗi model bằng OOF Utility, tie-break bằng AUPRC.
- Final 10-fold: giữ cố định `best_config`, tạo OOF prediction và báo cáo AUROC, AUPRC, Utility.

Checkpoint được lưu trong Modal Volume sau từng tuning candidate và từng final fold.


In [1]:
%uv pip install -q "scikit-learn>=1.5,<2" "lightgbm>=4.0,<5" "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.8"


Note: you may need to restart the kernel to use updated packages.


## 1. Environment and experiment configuration


In [2]:
from __future__ import annotations

import gc
import json
import time
import zipfile
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
np.random.seed(SEED)

VOLUME_ROOT = Path("/mnt/early-sepsis-xai")
INPUT_ZIP = VOLUME_ROOT / "training.zip"
WORK_DIR = Path("/tmp/early-sepsis-model-comparison")
RAW_DIR = WORK_DIR / "raw"
CHECKPOINT_DIR = VOLUME_ROOT / "lightgbm-v3-checkpoints"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_ZIP.is_file():
    raise FileNotFoundError(f"Upload training.zip to {INPUT_ZIP}")

HISTORY_HOURS = 5
TUNING_FOLDS = 5
FINAL_FOLDS = 10
LIGHTGBM_EARLY_STOPPING = 100
DECISION_TREE_FOLD_JOBS = 5
POSITIVE_WEIGHT = 40.0
SEARCH_ITERATIONS = {
    "Decision Tree": 10,
    "Random Forest": 10,
    "LightGBM": 20,
}

print(f"LightGBM {lgb.__version__}")
print(f"Dataset source: {INPUT_ZIP}")
for model_name, candidate_count in SEARCH_ITERATIONS.items():
    tuning_fits = TUNING_FOLDS * candidate_count
    print(f"{model_name}: {candidate_count} candidates → {tuning_fits:,} tuning fits + {FINAL_FOLDS} final fits")


LightGBM 4.7.0
Dataset source: /mnt/early-sepsis-xai/training.zip
Decision Tree: 10 candidates → 50 tuning fits + 10 final fits
Random Forest: 10 candidates → 50 tuning fits + 10 final fits
LightGBM: 20 candidates → 100 tuning fits + 10 final fits


## 2. Extract and index patient files


In [3]:
def extract_dataset(zip_path: Path, destination: Path) -> None:
    marker = destination / ".complete"
    if marker.exists():
        return
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)
    marker.touch()


def classify_source(patient_id: str) -> str:
    return "A" if int(patient_id[1:]) < 100_000 else "B"


def index_patient_files(root: Path) -> list[tuple[str, Path, str]]:
    records = []
    seen = set()
    for path in sorted(root.rglob("*.psv")):
        patient_id = path.stem
        if patient_id in seen:
            continue
        seen.add(patient_id)
        records.append((patient_id, path, classify_source(patient_id)))
    return records


extract_dataset(INPUT_ZIP, RAW_DIR)
patient_records = index_patient_files(RAW_DIR)
if len(patient_records) != 40_336:
    raise ValueError(f"Expected 40,336 patients, found {len(patient_records):,}")

source_counts = pd.Series([source for _, _, source in patient_records]).value_counts().sort_index()
print(f"Indexed {len(patient_records):,} patients")
display(source_counts.rename("patients").to_frame())


Indexed 40,336 patients


,patients
A,20336
B,20000


## 3. Reproduce preprocessing: Base + Mask, H = 5


In [4]:
VITAL_SIGNS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = [
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
    "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
    "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium",
    "Bilirubin_total", "TroponinI", "Hct", "Hgb", "PTT", "WBC",
    "Fibrinogen", "Platelets",
]
DEMOGRAPHICS = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
CLINICAL_VARS = VITAL_SIGNS + LABS + DEMOGRAPHICS
MASKED_VARS = VITAL_SIGNS + LABS
LOG_TRANSFORM_VARS = [
    "FiO2", "WBC", "HospAdmTime", "Alkalinephos", "AST", "TroponinI",
    "Bilirubin_total", "Creatinine", "O2Sat", "ICULOS", "Bilirubin_direct",
    "PTT", "Lactate", "Glucose", "BUN", "SaO2", "Magnesium",
    "Platelets", "Potassium", "Phosphate", "Calcium", "Fibrinogen",
    "PaCO2", "DBP", "MAP",
]

BASE_FEATURE_NAMES = CLINICAL_VARS + [f"{name}_mask" for name in MASKED_VARS]
FEATURE_NAMES = [
    f"{name}_t0" if lag == 0 else f"{name}_t-{lag}"
    for lag in range(HISTORY_HOURS)
    for name in BASE_FEATURE_NAMES
]
FEATURES_PER_HOUR = len(BASE_FEATURE_NAMES)

assert len(CLINICAL_VARS) == 40
assert len(MASKED_VARS) == 34
assert len(FEATURE_NAMES) == 370


def signed_log1p(frame: pd.DataFrame) -> pd.DataFrame:
    transformed = frame.copy()
    for column in LOG_TRANSFORM_VARS:
        values = transformed[column]
        transformed[column] = np.sign(values) * np.log1p(np.abs(values))
    return transformed


def fit_normalization_stats(records: list[tuple[str, Path, str]]) -> tuple[dict[str, dict[str, float]], dict[str, int]]:
    count = np.zeros(len(CLINICAL_VARS), dtype=np.int64)
    total = np.zeros(len(CLINICAL_VARS), dtype=np.float64)
    squared_total = np.zeros(len(CLINICAL_VARS), dtype=np.float64)
    row_counts = {}
    for patient_id, path, _ in records:
        raw = pd.read_csv(path, sep="|")
        row_counts[patient_id] = len(raw)
        values = signed_log1p(raw[CLINICAL_VARS]).to_numpy(dtype=np.float64)
        count += np.sum(~np.isnan(values), axis=0)
        total += np.nansum(values, axis=0)
        squared_total += np.nansum(np.square(values), axis=0)

    stats = {}
    for index, column in enumerate(CLINICAL_VARS):
        mean = total[index] / count[index]
        variance = squared_total[index] / count[index] - mean**2
        stats[column] = {"mean": float(mean), "std": float(np.sqrt(max(variance, 1e-12)))}
    return stats, row_counts


def preprocess_patient(raw: pd.DataFrame, stats: dict[str, dict[str, float]]) -> tuple[np.ndarray, np.ndarray]:
    mask = raw[MASKED_VARS].notna().to_numpy(dtype=np.float32)
    values = signed_log1p(raw[CLINICAL_VARS])
    for column in CLINICAL_VARS:
        values[column] = (values[column] - stats[column]["mean"]) / stats[column]["std"]
    return values.ffill().fillna(0.0).to_numpy(dtype=np.float32), mask


def add_lookback(features: np.ndarray, hours: int) -> np.ndarray:
    output = np.zeros((len(features), features.shape[1] * hours), dtype=np.float32)
    for lag in range(hours):
        start = lag * features.shape[1]
        if lag == 0:
            output[:, start:start + features.shape[1]] = features
        else:
            output[lag:, start:start + features.shape[1]] = features[:-lag]
    return output


## 4. Build the shared 370-feature model matrix


In [5]:
def build_dataset(
    records: list[tuple[str, Path, str]],
    stats: dict[str, dict[str, float]],
    row_counts: dict[str, int],
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    total_rows = sum(row_counts.values())
    X = np.empty((total_rows, len(FEATURE_NAMES)), dtype=np.float32)
    y = np.empty(total_rows, dtype=np.int8)
    groups = np.empty(total_rows, dtype="U8")
    sources = np.empty(total_rows, dtype="U1")

    offset = 0
    for patient_id, path, source in records:
        raw = pd.read_csv(path, sep="|")
        values, mask = preprocess_patient(raw, stats)
        end = offset + len(raw)
        X[offset:end] = add_lookback(np.column_stack((values, mask)), HISTORY_HOURS)
        y[offset:end] = raw["SepsisLabel"].to_numpy(dtype=np.int8)
        groups[offset:end] = patient_id
        sources[offset:end] = source
        offset = end
    return X, y, groups, sources


normalization_stats, row_counts = fit_normalization_stats(patient_records)
X, y, groups, sources = build_dataset(patient_records, normalization_stats, row_counts)
print(f"X={X.shape}, y={y.shape}, patients={np.unique(groups).size:,}")
print(f"Positive timestep rate: {y.mean():.4%}")


X=(1552210, 370), y=(1552210,), patients=40,336
Positive timestep rate: 1.7985%


## 5. Utility score, OOF metrics and model factories


In [6]:
UTILITY_PARAMS = {
    "dt_early": -12, "dt_optimal": -6, "dt_late": 3,
    "max_u_tp": 1.0, "min_u_fn": -2.0, "u_fp": -0.05,
}


def patient_slices(patient_groups: np.ndarray) -> list[slice]:
    starts = np.flatnonzero(np.r_[True, patient_groups[1:] != patient_groups[:-1]])
    ends = np.r_[starts[1:], len(patient_groups)]
    return [slice(start, end) for start, end in zip(starts, ends)]


PATIENT_SLICES = patient_slices(groups)


def prediction_utility(labels: np.ndarray, predictions: np.ndarray) -> float:
    p = UTILITY_PARAMS
    is_septic = bool(np.any(labels))
    t_sepsis = int(np.argmax(labels) - p["dt_optimal"]) if is_septic else np.inf
    m1 = p["max_u_tp"] / (p["dt_optimal"] - p["dt_early"])
    b1 = -m1 * p["dt_early"]
    m2 = -p["max_u_tp"] / (p["dt_late"] - p["dt_optimal"])
    b2 = -m2 * p["dt_late"]
    m3 = p["min_u_fn"] / (p["dt_late"] - p["dt_optimal"])
    b3 = -m3 * p["dt_optimal"]
    utility = 0.0
    for hour, predicted in enumerate(predictions):
        if hour > t_sepsis + p["dt_late"]:
            continue
        if is_septic and predicted:
            utility += max(m1 * (hour - t_sepsis) + b1, p["u_fp"]) if hour <= t_sepsis + p["dt_optimal"] else m2 * (hour - t_sepsis) + b2
        elif not is_septic and predicted:
            utility += p["u_fp"]
        elif is_septic and not predicted and hour > t_sepsis + p["dt_optimal"]:
            utility += m3 * (hour - t_sepsis) + b3
    return utility


def normalized_utility(probabilities: np.ndarray, threshold: float) -> float:
    observed = best = inaction = 0.0
    for patient_slice in PATIENT_SLICES:
        labels = y[patient_slice]
        predictions = (probabilities[patient_slice] >= threshold).astype(np.int8)
        observed += prediction_utility(labels, predictions)
        inaction += prediction_utility(labels, np.zeros_like(labels))
        best_predictions = np.zeros_like(labels)
        if np.any(labels):
            onset = int(np.argmax(labels) - UTILITY_PARAMS["dt_optimal"])
            start = max(0, onset + UTILITY_PARAMS["dt_early"])
            end = min(len(labels), onset + UTILITY_PARAMS["dt_late"] + 1)
            best_predictions[start:end] = 1
        best += prediction_utility(labels, best_predictions)
    return (observed - inaction) / (best - inaction)


def find_best_threshold(probabilities: np.ndarray) -> tuple[float, float]:
    candidates = np.unique(np.concatenate([
        np.geomspace(0.005, 0.05, 10),
        np.linspace(0.05, 0.95, 19),
    ]))
    scores = [(float(threshold), normalized_utility(probabilities, float(threshold))) for threshold in candidates]
    return max(scores, key=lambda item: item[1])


SEARCH_SPACES = {
    "Decision Tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [6, 10, 14, None],
        "min_samples_split": [10, 50, 200],
        "min_samples_leaf": [5, 20, 50, 100],
        "max_features": [None, "sqrt", 0.5],
        "ccp_alpha": [0.0, 1e-5, 1e-4],
    },
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [8, 12, 16],
        "min_samples_split": [10, 50, 200],
        "min_samples_leaf": [5, 20, 50],
        "max_features": ["sqrt", 0.25, 0.5],
        "max_samples": [0.6, 0.8],
        "ccp_alpha": [0.0, 1e-5],
    },
    "LightGBM": {
        "n_estimators": [1000, 2000, 3000],
        "learning_rate": [0.01, 0.02, 0.03],
        "num_leaves": [7, 15, 31],
        "max_depth": [-1, 4, 6],
        "min_child_samples": [20, 50, 100],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
        "reg_alpha": [0.0, 0.1],
        "reg_lambda": [0.1, 1.0, 5.0],
        "scale_pos_weight": [20.0, 40.0, 60.0],
    },
}


def make_estimator(model_name: str, params: dict, seed: int = SEED):
    if model_name == "Decision Tree":
        return DecisionTreeClassifier(
            **params,
            class_weight={0: 1.0, 1: POSITIVE_WEIGHT},
            random_state=seed,
        )
    if model_name == "Random Forest":
        return RandomForestClassifier(
            **params,
            bootstrap=True,
            class_weight={0: 1.0, 1: POSITIVE_WEIGHT},
            n_jobs=-1,
            random_state=seed,
        )
    if model_name == "LightGBM":
        lightgbm_params = {**params}
        lightgbm_params.setdefault("scale_pos_weight", POSITIVE_WEIGHT)
        return LGBMClassifier(
            **lightgbm_params,
            objective="binary",
            subsample_freq=1,
            n_jobs=-1,
            random_state=seed,
            verbosity=-1,
        )
    raise ValueError(f"Unknown model: {model_name}")


## 6. Five-fold tuning and final ten-fold evaluation


In [7]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def model_slug(model_name: str) -> str:
    return model_name.lower().replace(" ", "-")


def candidate_configs(model_name: str, model_index: int) -> list[dict]:
    return list(ParameterSampler(
        SEARCH_SPACES[model_name],
        n_iter=SEARCH_ITERATIONS[model_name],
        random_state=SEED + model_index,
    ))


def fit_predict_split(
    model_name: str,
    config: dict,
    train_rows: np.ndarray,
    valid_rows: np.ndarray,
    split_seed: int,
) -> tuple[np.ndarray, np.ndarray, dict]:
    model = make_estimator(model_name, config, seed=split_seed)
    if model_name == "LightGBM":
        model.fit(
            X[train_rows], y[train_rows],
            eval_X=X[valid_rows],
            eval_y=y[valid_rows],
            eval_metric="average_precision",
            callbacks=[lgb.early_stopping(LIGHTGBM_EARLY_STOPPING, verbose=False)],
        )
        best_iteration = int(model.best_iteration_)
    else:
        model.fit(X[train_rows], y[train_rows])
        best_iteration = None
    probability = model.predict_proba(X[valid_rows])[:, 1].astype(np.float32)
    metrics = {
        "auroc": float(roc_auc_score(y[valid_rows], probability)),
        "auprc": float(average_precision_score(y[valid_rows], probability)),
        "best_iteration": best_iteration,
    }
    del model
    gc.collect()
    return valid_rows, probability, metrics


TUNING_SPLITS = list(GroupKFold(n_splits=TUNING_FOLDS).split(np.zeros(len(y)), y, groups))
FINAL_SPLITS = list(GroupKFold(n_splits=FINAL_FOLDS).split(np.zeros(len(y)), y, groups))


def evaluate_candidate(model_name: str, config: dict, candidate_id: int) -> dict:
    started = time.perf_counter()
    oof_probability = np.full(len(y), np.nan, dtype=np.float32)
    if model_name == "Decision Tree":
        fold_outputs = Parallel(n_jobs=DECISION_TREE_FOLD_JOBS, prefer="threads")(
            delayed(fit_predict_split)(model_name, config, train_rows, valid_rows, SEED + fold)
            for fold, (train_rows, valid_rows) in enumerate(TUNING_SPLITS)
        )
    else:
        fold_outputs = [
            fit_predict_split(model_name, config, train_rows, valid_rows, SEED + fold)
            for fold, (train_rows, valid_rows) in enumerate(TUNING_SPLITS)
        ]

    fold_auroc = []
    fold_auprc = []
    best_iterations = []
    for valid_rows, probability, metrics in fold_outputs:
        oof_probability[valid_rows] = probability
        fold_auroc.append(metrics["auroc"])
        fold_auprc.append(metrics["auprc"])
        if metrics["best_iteration"] is not None:
            best_iterations.append(metrics["best_iteration"])

    threshold, utility = find_best_threshold(oof_probability)
    record = {
        "candidate": candidate_id,
        "tuning_utility": float(utility),
        "tuning_auroc": float(roc_auc_score(y, oof_probability)),
        "tuning_auprc": float(average_precision_score(y, oof_probability)),
        "fold_auroc_std": float(np.std(fold_auroc)),
        "fold_auprc_std": float(np.std(fold_auprc)),
        "threshold": float(threshold),
        "mean_best_iteration": float(np.mean(best_iterations)) if best_iterations else None,
        "seconds": float(time.perf_counter() - started),
        "params": json_ready(config),
    }
    del oof_probability, fold_outputs
    gc.collect()
    return record


def tuning_checkpoint_path(model_name: str) -> Path:
    return CHECKPOINT_DIR / f"{model_slug(model_name)}-tuning5.json"


def tune_model(model_name: str, model_index: int) -> tuple[pd.DataFrame, dict]:
    checkpoint_path = tuning_checkpoint_path(model_name)
    records = json.loads(checkpoint_path.read_text()) if checkpoint_path.is_file() else []
    completed = {int(record["candidate"]) for record in records}
    configs = candidate_configs(model_name, model_index)
    if records:
        print(f"Resuming {model_name}: {len(records)}/{len(configs)} candidates completed")

    for candidate_id, config in enumerate(configs, start=1):
        if candidate_id in completed:
            continue
        record = evaluate_candidate(model_name, config, candidate_id)
        records.append(record)
        records.sort(key=lambda item: item["candidate"])
        checkpoint_path.write_text(json.dumps(json_ready(records), indent=2))
        print(
            f"candidate {candidate_id:>2}/{len(configs)} | "
            f"Utility={record['tuning_utility']:.5f} | "
            f"AUROC={record['tuning_auroc']:.5f} | "
            f"AUPRC={record['tuning_auprc']:.5f} | "
            f"time={record['seconds'] / 60:.1f} min"
        )

    tuning_table = pd.DataFrame(records).sort_values(
        ["tuning_utility", "tuning_auprc"], ascending=False
    ).reset_index(drop=True)
    best_config = tuning_table.loc[0, "params"].copy()
    display(tuning_table.drop(columns="params").head(10))
    print("Selected best_config")
    display(pd.Series(best_config, name="value").to_frame())
    return tuning_table, best_config


def final_checkpoint_paths(model_name: str) -> tuple[Path, Path]:
    slug = model_slug(model_name)
    return CHECKPOINT_DIR / f"{slug}-final10-oof.npy", CHECKPOINT_DIR / f"{slug}-final10.json"


def run_final_cv(model_name: str, best_config: dict) -> tuple[pd.DataFrame, dict]:
    probability_path, metadata_path = final_checkpoint_paths(model_name)
    if probability_path.is_file() and metadata_path.is_file():
        oof_probability = np.load(probability_path)
        fold_records = json.loads(metadata_path.read_text())
        print(f"Resuming {model_name} final CV: {len(fold_records)}/{FINAL_FOLDS} folds completed")
    else:
        oof_probability = np.full(len(y), np.nan, dtype=np.float32)
        fold_records = []
    completed = {int(record["fold"]) for record in fold_records}

    for fold, (train_rows, valid_rows) in enumerate(FINAL_SPLITS):
        if fold in completed:
            continue
        started = time.perf_counter()
        valid_rows, probability, metrics = fit_predict_split(
            model_name, best_config, train_rows, valid_rows, SEED + fold
        )
        oof_probability[valid_rows] = probability
        fold_record = {
            "fold": fold,
            "auroc": metrics["auroc"],
            "auprc": metrics["auprc"],
            "best_iteration": metrics["best_iteration"],
            "seconds": float(time.perf_counter() - started),
        }
        fold_records.append(fold_record)
        fold_records.sort(key=lambda item: item["fold"])
        np.save(probability_path, oof_probability)
        metadata_path.write_text(json.dumps(json_ready(fold_records), indent=2))
        print(
            f"Fold {fold}: AUROC={fold_record['auroc']:.4f}, "
            f"AUPRC={fold_record['auprc']:.4f}, "
            f"time={fold_record['seconds'] / 60:.1f} min"
        )

    if np.isnan(oof_probability).any():
        raise RuntimeError(f"OOF predictions are incomplete for {model_name}")
    threshold, utility = find_best_threshold(oof_probability)
    summary = {
        "Model": model_name,
        "OOF AUROC": float(roc_auc_score(y, oof_probability)),
        "OOF AUPRC": float(average_precision_score(y, oof_probability)),
        "OOF Utility": float(utility),
        "Threshold": float(threshold),
        "Total hours": float(sum(record["seconds"] for record in fold_records) / 3600),
    }
    fold_table = pd.DataFrame(fold_records)
    display(fold_table)
    display(pd.Series(summary, name="value").to_frame())
    return fold_table, summary


## 7. Decision Tree: 5-fold tuning and final 10-fold

Thử 10 cấu hình bằng 5-fold OOF, chọn một `best_config`, sau đó giữ cố định cấu hình này cho final 10-fold.


In [8]:
display(pd.Series(SEARCH_SPACES["Decision Tree"], name="candidate values").to_frame())
decision_tree_tuning, decision_tree_best_config = tune_model("Decision Tree", model_index=0)
decision_tree_folds, decision_tree_summary = run_final_cv("Decision Tree", decision_tree_best_config)


,candidate values
criterion,"[gini, entropy]"
max_depth,"[6, 10, 14, None]"
min_samples_split,"[10, 50, 200]"
min_samples_leaf,"[5, 20, 50, 100]"
max_features,"[None, sqrt, 0.5]"
ccp_alpha,"[0.0, 1e-05, 0.0001]"


candidate  1/10 | Utility=0.26337 | AUROC=0.71742 | AUPRC=0.06678 | time=0.6 min
candidate  2/10 | Utility=0.32706 | AUROC=0.76649 | AUPRC=0.08355 | time=1.6 min
candidate  3/10 | Utility=0.27654 | AUROC=0.73683 | AUPRC=0.06693 | time=0.6 min
candidate  4/10 | Utility=0.25395 | AUROC=0.73007 | AUPRC=0.06783 | time=0.6 min
candidate  5/10 | Utility=0.25395 | AUROC=0.73006 | AUPRC=0.06777 | time=0.6 min
candidate  6/10 | Utility=0.32058 | AUROC=0.76392 | AUPRC=0.08495 | time=1.6 min
candidate  7/10 | Utility=0.25217 | AUROC=0.72960 | AUPRC=0.06610 | time=0.6 min
candidate  8/10 | Utility=0.32058 | AUROC=0.76392 | AUPRC=0.08495 | time=1.6 min
candidate  9/10 | Utility=0.26130 | AUROC=0.71421 | AUPRC=0.06510 | time=0.6 min
candidate 10/10 | Utility=0.28416 | AUROC=0.74073 | AUPRC=0.06760 | time=0.6 min


,candidate,tuning_utility,tuning_auroc,tuning_auprc,fold_auroc_std,fold_auprc_std,threshold,mean_best_iteration,seconds
0,2,0.327060,0.766493,0.083550,0.007067,0.004235,0.50,None,96.094624
1,6,0.320576,0.763922,0.084949,0.007225,0.004335,0.50,None,98.330071
2,8,0.320576,0.763922,0.084949,0.007225,0.004335,0.50,None,96.588215
3,10,0.284162,0.740729,0.067597,0.018646,0.007458,0.60,None,38.838217
4,3,0.276542,0.736830,0.066926,0.017545,0.006994,0.60,None,38.651409
5,1,0.263368,0.717415,0.066781,0.014398,0.002927,0.60,None,38.597202
6,9,0.261301,0.714207,0.065097,0.017904,0.008954,0.55,None,38.488777
7,4,0.253946,0.730068,0.067835,0.012995,0.009019,0.50,None,36.485022
8,5,0.253946,0.730055,0.067775,0.012982,0.009002,0.50,None,36.683928
9,7,0.252168,0.729598,0.066099,0.013884,0.008601,0.50,None,37.011939


Selected best_config


,value
min_samples_split,10
min_samples_leaf,5
max_features,None
max_depth,6
criterion,entropy
ccp_alpha,0.0001


Fold 0: AUROC=0.7457, AUPRC=0.0751, time=1.2 min
Fold 1: AUROC=0.7671, AUPRC=0.0874, time=1.2 min
Fold 2: AUROC=0.7729, AUPRC=0.0894, time=1.2 min
Fold 3: AUROC=0.7791, AUPRC=0.0838, time=1.2 min
Fold 4: AUROC=0.7602, AUPRC=0.0776, time=1.2 min
Fold 5: AUROC=0.7836, AUPRC=0.0840, time=1.2 min
Fold 6: AUROC=0.7485, AUPRC=0.0786, time=1.2 min
Fold 7: AUROC=0.7796, AUPRC=0.0878, time=1.2 min
Fold 8: AUROC=0.7729, AUPRC=0.0878, time=1.2 min
Fold 9: AUROC=0.7524, AUPRC=0.0845, time=1.2 min


,fold,auroc,auprc,best_iteration,seconds
0,0,0.745650,0.075140,None,71.152463
1,1,0.767129,0.087413,None,71.373060
2,2,0.772947,0.089378,None,73.766285
3,3,0.779123,0.083831,None,72.014185
4,4,0.760171,0.077602,None,72.924632
5,5,0.783627,0.083957,None,71.032248
6,6,0.748504,0.078571,None,71.827865
7,7,0.779634,0.087767,None,71.668098
8,8,0.772863,0.087835,None,73.452013
9,9,0.752372,0.084473,None,71.660565


,value
Model,Decision Tree
OOF AUROC,0.767208
OOF AUPRC,0.085995
OOF Utility,0.322331
Threshold,0.55
Total hours,0.200242


## 8. Random Forest: 5-fold tuning and final 10-fold

Thử 10 cấu hình bằng 5-fold OOF, chọn một `best_config`, sau đó giữ cố định cấu hình này cho final 10-fold.


In [9]:
display(pd.Series(SEARCH_SPACES["Random Forest"], name="candidate values").to_frame())
random_forest_tuning, random_forest_best_config = tune_model("Random Forest", model_index=1)
random_forest_folds, random_forest_summary = run_final_cv("Random Forest", random_forest_best_config)


,candidate values
n_estimators,"[100, 200]"
max_depth,"[8, 12, 16]"
min_samples_split,"[10, 50, 200]"
min_samples_leaf,"[5, 20, 50]"
max_features,"[sqrt, 0.25, 0.5]"
max_samples,"[0.6, 0.8]"
ccp_alpha,"[0.0, 1e-05]"


candidate  1/10 | Utility=0.35342 | AUROC=0.82116 | AUPRC=0.09239 | time=13.1 min
candidate  2/10 | Utility=0.33122 | AUROC=0.81161 | AUPRC=0.08835 | time=24.9 min
candidate  3/10 | Utility=0.36034 | AUROC=0.81598 | AUPRC=0.09260 | time=5.0 min
candidate  4/10 | Utility=0.35079 | AUROC=0.81993 | AUPRC=0.09130 | time=30.0 min
candidate  5/10 | Utility=0.36967 | AUROC=0.81817 | AUPRC=0.09137 | time=2.7 min
candidate  6/10 | Utility=0.34861 | AUROC=0.81970 | AUPRC=0.09051 | time=15.2 min
candidate  7/10 | Utility=0.36984 | AUROC=0.81819 | AUPRC=0.09001 | time=2.6 min
candidate  8/10 | Utility=0.35735 | AUROC=0.81029 | AUPRC=0.08643 | time=3.6 min
candidate  9/10 | Utility=0.36717 | AUROC=0.81747 | AUPRC=0.09350 | time=8.6 min
candidate 10/10 | Utility=0.30878 | AUROC=0.79603 | AUPRC=0.08186 | time=2.2 min


,candidate,tuning_utility,tuning_auroc,tuning_auprc,fold_auroc_std,fold_auprc_std,threshold,mean_best_iteration,seconds
0,7,0.369844,0.818189,0.090010,0.005741,0.003648,0.40,None,158.930861
1,5,0.369672,0.818165,0.091369,0.005316,0.003556,0.40,None,163.345683
2,9,0.367174,0.817472,0.093497,0.007548,0.003944,0.40,None,516.656533
3,3,0.360339,0.815983,0.092604,0.007642,0.003331,0.40,None,297.864120
4,8,0.357351,0.810291,0.086433,0.005987,0.002275,0.35,None,218.422441
5,1,0.353417,0.821158,0.092392,0.005275,0.003593,0.25,None,786.242822
6,4,0.350794,0.819928,0.091300,0.005196,0.002291,0.25,None,1797.285149
7,6,0.348610,0.819701,0.090509,0.006305,0.003046,0.25,None,913.764516
8,2,0.331224,0.811606,0.088352,0.005921,0.002580,0.20,None,1496.607324
9,10,0.308781,0.796031,0.081863,0.006360,0.002979,0.30,None,132.563648


Selected best_config


,value
n_estimators,200
min_samples_split,200
min_samples_leaf,20
max_samples,0.6
max_features,sqrt
max_depth,8
ccp_alpha,0.0


Fold 0: AUROC=0.8075, AUPRC=0.0857, time=0.5 min
Fold 1: AUROC=0.8116, AUPRC=0.0914, time=0.5 min
Fold 2: AUROC=0.8135, AUPRC=0.0999, time=0.5 min
Fold 3: AUROC=0.8318, AUPRC=0.0876, time=0.5 min
Fold 4: AUROC=0.8178, AUPRC=0.0900, time=0.5 min
Fold 5: AUROC=0.8271, AUPRC=0.0982, time=0.5 min
Fold 6: AUROC=0.8182, AUPRC=0.0938, time=0.5 min
Fold 7: AUROC=0.8228, AUPRC=0.0956, time=0.5 min
Fold 8: AUROC=0.8277, AUPRC=0.1056, time=0.5 min
Fold 9: AUROC=0.8025, AUPRC=0.0898, time=0.5 min


,fold,auroc,auprc,best_iteration,seconds
0,0,0.807540,0.085678,None,28.439996
1,1,0.811578,0.091404,None,29.888085
2,2,0.813546,0.099944,None,31.974617
3,3,0.831768,0.087637,None,30.584857
4,4,0.817820,0.089984,None,29.034687
5,5,0.827084,0.098189,None,29.450890
6,6,0.818208,0.093835,None,29.803228
7,7,0.822795,0.095554,None,30.031350
8,8,0.827657,0.105626,None,32.035562
9,9,0.802499,0.089795,None,29.536370


,value
Model,Random Forest
OOF AUROC,0.817759
OOF AUPRC,0.091672
OOF Utility,0.36706
Threshold,0.4
Total hours,0.08355


## 9. LightGBM: 5-fold tuning and final 10-fold

Thử 20 cấu hình bằng 5-fold OOF với early stopping, chọn một `best_config`, sau đó giữ cố định cấu hình này cho final 10-fold.


In [8]:
display(pd.Series(SEARCH_SPACES["LightGBM"], name="candidate values").to_frame())
lightgbm_tuning, lightgbm_best_config = tune_model("LightGBM", model_index=2)
lightgbm_folds, lightgbm_summary = run_final_cv("LightGBM", lightgbm_best_config)


,candidate values
n_estimators,"[1000, 2000, 3000]"
learning_rate,"[0.01, 0.02, 0.03]"
num_leaves,"[7, 15, 31]"
max_depth,"[-1, 4, 6]"
min_child_samples,"[20, 50, 100]"
subsample,"[0.8, 1.0]"
colsample_bytree,"[0.7, 0.9, 1.0]"
reg_alpha,"[0.0, 0.1]"
reg_lambda,"[0.1, 1.0, 5.0]"
scale_pos_weight,"[20.0, 40.0, 60.0]"


candidate  1/20 | Utility=0.32290 | AUROC=0.78835 | AUPRC=0.08228 | time=5.2 min
candidate  2/20 | Utility=0.28252 | AUROC=0.75452 | AUPRC=0.06362 | time=3.5 min
candidate  3/20 | Utility=0.31725 | AUROC=0.76858 | AUPRC=0.07126 | time=4.9 min
candidate  4/20 | Utility=0.32300 | AUROC=0.77039 | AUPRC=0.08102 | time=7.3 min
candidate  5/20 | Utility=0.32779 | AUROC=0.78928 | AUPRC=0.07994 | time=5.3 min
candidate  6/20 | Utility=0.33084 | AUROC=0.77318 | AUPRC=0.08862 | time=4.8 min
candidate  7/20 | Utility=0.30323 | AUROC=0.73696 | AUPRC=0.08202 | time=4.7 min
candidate  8/20 | Utility=0.13544 | AUROC=0.72475 | AUPRC=0.06201 | time=4.7 min
candidate  9/20 | Utility=0.33058 | AUROC=0.76976 | AUPRC=0.08313 | time=4.4 min
candidate 10/20 | Utility=0.28204 | AUROC=0.75459 | AUPRC=0.06471 | time=3.4 min
candidate 11/20 | Utility=0.31178 | AUROC=0.76533 | AUPRC=0.08096 | time=7.5 min
candidate 12/20 | Utility=0.31315 | AUROC=0.77536 | AUPRC=0.07928 | time=5.6 min
candidate 13/20 | Utility=0.

,candidate,tuning_utility,tuning_auroc,tuning_auprc,fold_auroc_std,fold_auprc_std,threshold,mean_best_iteration,seconds
0,6,0.330835,0.773181,0.088620,0.007579,0.001941,0.029974,3.8,286.968756
1,9,0.330576,0.769756,0.083129,0.008976,0.001963,0.029974,3.6,262.783599
2,5,0.327794,0.789280,0.079943,0.005779,0.003587,0.029974,2.0,318.307890
3,4,0.323004,0.770387,0.081021,0.004341,0.001554,0.050000,1.0,440.725459
4,1,0.322904,0.788351,0.082276,0.005820,0.001246,0.029974,2.0,314.497743
5,20,0.318696,0.768601,0.071154,0.005873,0.000873,0.050000,1.0,316.025114
6,13,0.317931,0.735176,0.078632,0.004074,0.001778,0.029974,1.0,290.163090
7,3,0.317252,0.768579,0.071258,0.004858,0.000472,0.038713,1.0,295.425043
8,15,0.317114,0.783870,0.072475,0.007966,0.000385,0.038713,1.0,542.071045
9,14,0.315022,0.773070,0.079134,0.007764,0.001466,0.029974,1.0,348.244868


Selected best_config


,value
subsample,0.80
scale_pos_weight,20.00
reg_lambda,1.00
reg_alpha,0.10
num_leaves,15.00
n_estimators,1000.00
min_child_samples,20.00
max_depth,4.00
learning_rate,0.01
colsample_bytree,1.00


Fold 0: AUROC=0.7651, AUPRC=0.0904, time=0.8 min
Fold 1: AUROC=0.7672, AUPRC=0.0849, time=0.7 min
Fold 2: AUROC=0.7701, AUPRC=0.0895, time=0.8 min
Fold 3: AUROC=0.7880, AUPRC=0.0965, time=0.8 min
Fold 4: AUROC=0.7600, AUPRC=0.0894, time=0.8 min
Fold 5: AUROC=0.7765, AUPRC=0.0921, time=0.7 min
Fold 6: AUROC=0.7647, AUPRC=0.0804, time=0.7 min
Fold 7: AUROC=0.7922, AUPRC=0.0952, time=0.7 min
Fold 8: AUROC=0.7804, AUPRC=0.0925, time=0.7 min
Fold 9: AUROC=0.7844, AUPRC=0.0886, time=0.7 min


,fold,auroc,auprc,best_iteration,seconds
0,0,0.765059,0.090423,4,46.429109
1,1,0.767231,0.084912,3,44.755024
2,2,0.770142,0.089520,4,46.801202
3,3,0.787997,0.096496,4,47.738118
4,4,0.760025,0.089387,4,46.049533
5,5,0.776505,0.092107,4,43.287016
6,6,0.764740,0.080414,3,41.138603
7,7,0.792203,0.095195,4,43.221457
8,8,0.780407,0.092517,3,42.214016
9,9,0.784393,0.088647,3,42.153523


,value
Model,LightGBM
OOF AUROC,0.771546
OOF AUPRC,0.086185
OOF Utility,0.331667
Threshold,0.029974
Total hours,0.123274


## 10. Final comparison


In [2]:
comparison = pd.DataFrame([
    {"Model": "XGBoost", "OOF AUROC": 0.831129, "OOF AUPRC": 0.108687, "OOF Utility": 0.400953},
    {"Model": "Decision Tree", "OOF AUROC": 0.767208, "OOF AUPRC": 0.085995, "OOF Utility": 0.322331},
    {"Model": "Random Forest", "OOF AUROC": 0.817759, "OOF AUPRC": 0.091672, "OOF Utility": 0.367060},
    {"Model": "LightGBM", "OOF AUROC": 0.771546, "OOF AUPRC": 0.086185, "OOF Utility": 0.331667},
]).sort_values("OOF Utility", ascending=False).reset_index(drop=True)
display(comparison)

,Model,OOF AUROC,OOF AUPRC,OOF Utility
0,XGBoost,0.831129,0.108687,0.400953
1,Random Forest,0.817759,0.091672,0.367060
2,LightGBM,0.771546,0.086185,0.331667
3,Decision Tree,0.767208,0.085995,0.322331
